In [ ]:
# https://www.kaggle.com/competitions/store-sales-time-series-forecasting
# Score: 0.39350

In [ ]:
import sys
sys.path.insert(0, '/Volumes/Crucial_X9_2TB/Programme/PROGR-254')

import datetime as dt
import numpy as np
from pathlib import Path
import polars as pl
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from panel_forecaster import PanelConfig, PanelForecaster

# ─── 1) Daten laden ──────────────────────────────────────────────
path = Path("./data/")
train    = pl.read_csv(path / "train.csv",           try_parse_dates=True)
test     = pl.read_csv(path / "test.csv",            try_parse_dates=True)
stores   = pl.read_csv(path / "stores.csv")
oil_pl   = pl.read_csv(path / "oil.csv",             try_parse_dates=True) \
             .rename({"dcoilwtico": "oil_price"})
holidays = pl.read_csv(path / "holidays_events.csv", try_parse_dates=True)

# ─── 2) Stores mergen ────────────────────────────────────────────
train = train.join(stores, on="store_nbr", how="left")
test  = test.join(stores,  on="store_nbr", how="left")

# ─── 3) Oil ffill/bfill ──────────────────────────────────────────
oil_pl = oil_pl.with_columns(
    pl.col("oil_price").forward_fill().backward_fill()
)

# ─── 4) Holiday- und Earthquake-Listen ───────────────────────────
holidays_list = (
    holidays
    .filter((pl.col("transferred") != True) & (pl.col("locale") == "National"))
    ["date"].to_list()
)
transferred_list = holidays.filter(pl.col("transferred") == True)["date"].to_list()
eq_start = dt.date(2016, 4, 16)
eq_dates = [eq_start + dt.timedelta(days=i) for i in range(61)]

# ─── 5) days_since_earthquake (kontinuierlich, geclippt ≥ 0) ─────
train = train.with_columns(
    (pl.col("date").cast(pl.Date) - pl.lit(eq_start)).dt.total_days()
    .clip(lower_bound=0).alias("days_since_earthquake")
)
test = test.with_columns(
    (pl.col("date").cast(pl.Date) - pl.lit(eq_start)).dt.total_days()
    .clip(lower_bound=0).alias("days_since_earthquake")
)

# ─── 6) Config ───────────────────────────────────────────────────
cfg = PanelConfig(
    date_col="date", target_col="sales",
    entity_cols=["store_nbr", "family"],
    categorical_cols=["store_nbr", "family", "city", "state", "type"],
    target_encode_cols=["family"],
    exogenous=[{
        "frame": oil_pl,
        "rolling": {"oil_price": [7, 28]},
        "diff":    ["oil_price"],
    }],
    event_dates={
        "is_holiday":             holidays_list,
        "is_transferred_holiday": transferred_list,
        "is_post_earthquake":     eq_dates,
    },
    horizon=16,
    lags=(16, 17, 18, 19, 20, 21, 28, 30, 35, 60, 364, 371),
    rollings=(7, 14, 28, 60),
    rolling_aggs=("mean", "std", "median"),
    lag_aggregates={"mean_same_dow_4w": (21, 28, 35, 42)},
    extra_rollings=((364, 7, "mean"), (364, 28, "mean")),
    running_count_cols=["onpromotion"],
    use_hist_mean=True,
    use_zero_features=True,
    use_payday=True,
)

# ─── 7) Fit + Predict ────────────────────────────────────────────
train = train.drop("id")

custom_models = [
    ("lgbm", LGBMRegressor,    dict(verbose=-1, n_estimators=500, num_leaves=127, learning_rate=0.03, min_child_samples=20, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5)),
    ("xgb",  XGBRegressor,     dict(verbosity=0, n_estimators=500, max_depth=7, learning_rate=0.03)),
    ("cat",  CatBoostRegressor, dict(verbose=0, iterations=500, learning_rate=0.05, depth=6)),
]

model = PanelForecaster(cfg, models=custom_models, per_entity_col="family",
                        log_target=True, n_seeds=7)
model.fit(train, test=test)
preds = model.predict(test)

# ─── 8) Zero-Sales-Override ──────────────────────────────────────
cutoff = train["date"].max() - dt.timedelta(days=60)
dead = (
    train.filter(pl.col("date") >= cutoff)
         .group_by(["store_nbr", "family"])
         .agg(pl.col("sales").sum().alias("s"))
         .filter(pl.col("s") == 0)
         .select(["store_nbr", "family"])
)
dead_ids = test.join(dead, on=["store_nbr", "family"], how="inner")["id"].to_list()
mask = np.isin(test["id"].to_numpy(), dead_ids)
preds[mask] = 0
print(f"Zero-Override: {dead.height} Serien, {mask.sum()} rows auf 0 gesetzt")

# ─── 9) Submission ───────────────────────────────────────────────
sub = pl.DataFrame({"id": test["id"], "sales": preds})
sub.write_csv(path / "submission.csv")
print(sub.describe())
print(f"\n✅ {path}/submission.csv")